In [2]:
#pip install pyspark

In [3]:
# Big Data

In [5]:
from pyspark import SparkContext , SparkConf
import collections
sc = SparkContext()
rdd = sc.parallelize([3,4,56,7,4,5,6])
sq = rdd.map(lambda x : x*x)
print(sq.collect())
sc.stop()
# yukarıdaki sayıların karesini alıp yazdırma
# binlerce satır veri aynı anda gelirken gerçek zamanlı anliz etmemizi spark sağlar

[9, 16, 3136, 49, 16, 25, 36]


In [11]:
# hata aldığım için hocanınkinden farklı oldu ama sonuçlar aynı
from pyspark import SparkContext, SparkConf # configuration
import collections

# Check if a SparkContext already exists, otherwise create a new one
sc = SparkContext.getOrCreate(SparkConf().setMaster("local").setAppName("RatingHistogram"))

lines=sc.textFile('u.data')
ratings=lines.map(lambda x:x.split()[2])
result=ratings.countByValue()
sortedResults=collections.OrderedDict(sorted(result.items()))
for key,value in sortedResults.items():
    print("%s %i"%(key,value))
# sc.stop() # Do not stop the context if it might be needed later

1 6111
2 11370
3 27145
4 34174
5 21203


In [25]:
'''
from pyspark import SparkContext, SparkConf # configuration
import collections

conf=SparkConf().setMaster("local").setAppName("MinTempratures")
sc=SparkContext(conf=conf)


def parseline(line):
  fields = line.split(',')
  stationID = fields[0]
  entryType = fields[2]
  temperature = float(fields[3]) * 0.1 * (9.0 / 5.0) + 32.0
lines = sc.textFile("1800.csv")
parsedLines = lines.map(parseline)
minTemps = parsedLines.filter(lambda x: "TMIN" in x[1])
stationTemps = minTemps.map(lambda x: (x[0], x[2]))
minTemps = stationTemps.reduceByKey(lambda x, y: min(x,y))
results = minTemps.collect();

for result in results:
    print(result[0] + "\t{:.2f}F".format(result[1]))

sc.stop()

'''

# Chat gpt düzeltti
from pyspark import SparkContext, SparkConf
import collections

conf = SparkConf().setMaster("local").setAppName("MinTemperatures")
sc = SparkContext(conf=conf)

def parseline(line):
    try:
        fields = line.split(',')
        stationID = fields[0]
        entryType = fields[2]
        # Orijinal veri 0.1 °C biriminde -> önce °C, sonra °F
        temp_c = float(fields[3]) * 0.1
        temp_f = temp_c * 9.0 / 5.0 + 32.0
        return (stationID, entryType, temp_f)
    except Exception:
        return None  # bozuk satır

# ---- Pipeline ----
lines = sc.textFile("1800.csv")

parsedLines = (
    lines
    .map(parseline)
    .filter(lambda x: x is not None)           # None'ları at
)

minTempsOnly = parsedLines.filter(lambda x: x[1] == "TMIN")  # sadece TMIN
stationTemps = minTempsOnly.map(lambda x: (x[0], x[2]))      # (istasyon, sıcaklık°F)
minTempsByStation = stationTemps.reduceByKey(lambda a, b: a if a < b else b)

results = minTempsByStation.collect()

for station, temp in results:
    print(f"{station}\t{temp:.2f}F")

sc.stop()

ITE00100554	5.36F
EZE00100082	7.70F


In [24]:
sc.stop()

In [ ]:
# hangi kelimenin kaç kere geçtiğini buldu
from pyspark import SparkContext, SparkConf # configuration
import collections

# Check if a SparkContext already exists, otherwise create a new one
sc = SparkContext.getOrCreate(SparkConf().setMaster("local").setAppName("WordCount"))

input=sc.textFile("book.txt")
words=input.flatMap(lambda x:x.split())
wordCounts=words.countByValue()

for word,count in wordCounts.items():
    cleanWord=word.encode('ascii','ignore')
    if(cleanWord):
        print(cleanWord.decode()+" "+str(count))
# sc.stop() # Do not stop the context if it might be needed later

In [32]:
''' hocanın kod çalışmadı
from pyspark import SparkConf, SparkContext
conf = SparkConf().setMaster("local").setAppName("PopularHero")
sc = SparkContext(conf = conf)
def countCoOccurences(line):
    elements = line.split()
    return (int(elements[0]), len(elements) - 1)
def parseNames(line):
    fields = line.split('\"')
    return (int(fields[0]), fields[1].encode("utf8"))
names = sc.textFile("Marvel-names.txt")
namesRdd = names.map(parseNames)
lines = sc.textFile("Marvel-graph.txt")
pairings = lines.map(countCoOccurences)
totalFriendsByCharacter = pairings.reduceByKey(lambda x, y : x + y)
flipped = totalFriendsByCharacter.map(lambda xy : (xy[1], xy[0]))
mostPopular = flipped.max()
mostPopularName = namesRdd.lookup(mostPopular[1])[0]
print(str(mostPopularName) + " is the most popular superhero, with " + str(mostPopular[0]) + \
      " co-appearances.")
sc.stop()
'''
from pyspark import SparkConf, SparkContext
# Check if a SparkContext already exists, otherwise create a new one
sc = SparkContext.getOrCreate(SparkConf().setMaster("local").setAppName("PopularHero"))

def countCoOccurences(line):
    elements = line.split()
    return (int(elements[0]), len(elements) - 1)
def parseNames(line):
    fields = line.split('\"')
    return (int(fields[0]), fields[1].encode("utf8"))
names = sc.textFile("Marvel-names.txt")
namesRdd = names.map(parseNames)
lines = sc.textFile("Marvel-graph.txt")
pairings = lines.map(countCoOccurences)
totalFriendsByCharacter = pairings.reduceByKey(lambda x, y : x + y)
flipped = totalFriendsByCharacter.map(lambda xy : (xy[1], xy[0]))
mostPopular = flipped.max()
mostPopularName = namesRdd.lookup(mostPopular[1])[0]
print(str(mostPopularName) + " is the most popular superhero, with " + str(mostPopular[0]) + \
      " co-appearances.")
# sc.stop() # Do not stop the context if it might be needed later

b'CAPTAIN AMERICA' is the most popular superhero, with 1933 co-appearances.


In [33]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
spark = SparkSession.builder.appName("PimaIndianClassifier").getOrCreate()
data= spark.read.csv("pima-indians-diabetes.csv", inferSchema=True, header=True)
predictors = data.columns[:-1]
assembler = VectorAssembler(inputCols=predictors, outputCol="features")
data = assembler.transform(data).select( "features", "Outcome")
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)
lr = LogisticRegression(labelCol="Outcome", featuresCol="features")
model = lr.fit(train_data)
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="Outcome")
accuracy = evaluator.evaluate(predictions)
print( "Accuracy: ", accuracy)
#bunla ilgili ödeve gelecek tek değişiklik data dosyası ve select( "features", "Outcome") burdaki outcome değişecek

Accuracy:  0.8546265328874024
